# 02 - Text Analysis

Lightweight linguistic look at the **ClearFairy Cognitive Decision Steps** dataset.

What this notebook covers:
- Token and sentence counts per field
- Most frequent content words in `decision_and_actions` and `rationale`
- Action-verb categories (heuristic, keyword-based)
- How action categories differ between the two task types

In [ ]:
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from scripts.load import load_dataset

df = pd.DataFrame(load_dataset())
df['has_rationale'] = df['rationale'].str.strip().astype(bool)
len(df)

## Token and sentence counts

Quick whitespace-tokenized counts. Not linguistically rigorous, but enough for a sense of scale.

In [ ]:
def n_tokens(text: str) -> int:
    return len(text.split()) if text else 0

def n_sentences(text: str) -> int:
    if not text:
        return 0
    return len([s for s in re.split(r'[.!?]+', text) if s.strip()])

for col in ['decision_and_actions', 'rationale', 'progression']:
    df[f'{col}_tokens'] = df[col].apply(n_tokens)
    df[f'{col}_sents'] = df[col].apply(n_sentences)

summary = pd.DataFrame({
    'tokens_mean':  [df[f'{c}_tokens'].mean() for c in ['decision_and_actions', 'rationale', 'progression']],
    'tokens_total': [df[f'{c}_tokens'].sum()  for c in ['decision_and_actions', 'rationale', 'progression']],
    'sents_mean':   [df[f'{c}_sents'].mean()  for c in ['decision_and_actions', 'rationale', 'progression']],
}, index=['decision_and_actions', 'rationale', 'progression']).round(1)
summary

## Most frequent content words

After stripping a small stoplist, what words show up the most?

In [ ]:
STOPWORDS = set('''
a an the and or but if then else of in on at to from for by with into onto upon
is are was were be been being am do does did doing have has had having will would
shall should can could may might must this that these those there here it its
they them their theirs his her hers him she he we us our ours you your yours i me my
as so than such not no nor only also more most less least very much many few one
two three four first second new old which who whom whose what when where why how
while during after before between within without across through about over under
designer design designed text element added create created creating change changed
step decision action actions decisions rationale progression
'''.split())

WORD_RE = re.compile(r"[A-Za-z][A-Za-z'-]+")

def words(text: str) -> list[str]:
    return [w.lower() for w in WORD_RE.findall(text or '') if w.lower() not in STOPWORDS and len(w) > 2]

for col in ['decision_and_actions', 'rationale']:
    counter = Counter()
    for txt in df[col]:
        counter.update(words(txt))
    print(f'\n=== Top 20 in {col} ===')
    for w, n in counter.most_common(20):
        print(f'  {w:<20} {n}')

## Action-verb categories (heuristic)

We tag each step with one or more *action categories* by checking for keyword matches in `decision_and_actions`. The categories are coarse but useful for getting a feel for what kinds of edits dominate each task type.

Note: a step can match multiple categories.

In [ ]:
ACTION_CATEGORIES = {
    'create':     ['created', 'added', 'inserted', 'introduced'],
    'delete':     ['removed', 'deleted'],
    'resize':     ['resized', 'width', 'height', 'dimensions'],
    'reposition': ['moved', 'positioned', 'aligned', 'x-position', 'y-position', 'repositioned'],
    'typography': ['font', 'typeface', 'letter', 'line height', 'bold', 'italic', 'character'],
    'color':      ['color', 'fill', 'stroke', 'opacity', 'gradient'],
    'layout':     ['frame', 'group', 'auto layout', 'padding', 'spacing', 'gap', 'columns', 'rows'],
    'content':    ['label', 'rename', 'renamed', 'characters', 'text content', 'changed the text'],
    'image':      ['image', 'photo', 'picture', 'icon'],
}

def categorize(text: str) -> set[str]:
    t = (text or '').lower()
    return {cat for cat, kws in ACTION_CATEGORIES.items() if any(kw in t for kw in kws)}

df['categories'] = df['decision_and_actions'].apply(categorize)
df['n_categories'] = df['categories'].apply(len)
df[['participant_id', 'task_type', 'categories']].head(5)

In [ ]:
# Per-task category coverage (% of steps in that task containing the category)
rows = []
for task in ['lab_website', 'shopping_site']:
    sub = df[df.task_type == task]
    for cat in ACTION_CATEGORIES:
        rows.append({
            'task_type': task,
            'category': cat,
            'pct_steps': sub['categories'].apply(lambda s: cat in s).mean() * 100,
        })
cat_df = pd.DataFrame(rows)
cat_pivot = cat_df.pivot(index='category', columns='task_type', values='pct_steps').round(1)
cat_pivot['delta'] = (cat_pivot['shopping_site'] - cat_pivot['lab_website']).round(1)
cat_pivot.sort_values('delta', ascending=False)

In [ ]:
ax = cat_pivot[['lab_website', 'shopping_site']].plot.barh(
    figsize=(8, 4),
    title='Action-category coverage by task type (% of steps)',
    color=['#4C78A8', '#F58518'],
)
ax.set_xlabel('% of steps in the task')
ax.set_ylabel('')
ax.figure.tight_layout()

## Steps that match no category

Useful sanity check: how many steps fall through every keyword filter? These are candidates for improving the heuristic, or for reviewing manually.

In [ ]:
uncategorized = df[df.n_categories == 0]
print(f'{len(uncategorized)} of {len(df)} steps ({len(uncategorized)/len(df)*100:.1f}%) match no category')
uncategorized[['participant_id', 'task_type', 'step_index', 'decision_and_actions']].head(5)